# Mô hình gu: ALS trên DS300_data

Notebook này **độc lập hoàn toàn**: chỉ cần attach dataset `DS300_data` có `train.csv` và `test.csv`, không cần upload thư mục `src`.

- Chọn cấu hình bằng validation tách từ `train.csv`.
- `test.csv` chỉ được dùng một lần ở bước cuối.
- So sánh `Popularity`, ALS-binary, ALS-log-playcount và ALS-engagement.
- Ghi CSV/JSON vào `/kaggle/working/runs/taste_10k_01/`; tải ở tab **Output** khi chạy xong.

In [ ]:
# Cấu hình thí nghiệm
from pathlib import Path
import os, json, shutil, hashlib, math, sys
import numpy as np
import pandas as pd

# Path dataset DS300_data em đã attach trên Kaggle.
DATA_DIR = Path('/kaggle/input/datasets/vminhbin/ds300-data')
TRAIN_PATH, TEST_PATH = DATA_DIR / 'train.csv', DATA_DIR / 'test.csv'

SEED = 42
# Bản notebook thuần NumPy: dùng 1.000 user để chạy ổn định trên Kaggle.
# Kết quả 10.000 user chạy bằng implicit trên máy local là kết quả chính để báo cáo.
MAX_TRAIN_USERS = 1_000
MAX_EVAL_USERS = 300
FACTORS = 16
ITERATIONS = 8
ALS_ALPHA = 10.0
REGULARIZATION = 0.1
WEIGHTINGS = ['binary', 'log', 'engagement']
RUN_NAME = 'taste_10k_01'
OUT_DIR = Path('/kaggle/working/runs') / RUN_NAME

print('DATA_DIR =', DATA_DIR)
print('TRAIN    =', TRAIN_PATH)
print('TEST     =', TEST_PATH)

In [ ]:
# Không cài implicit: Kaggle đang dùng Python không có wheel tương thích cho package này.
# Notebook dùng NumPy reference implementation của Hu-style weighted ALS.
# Nó chậm hơn implicit, vì vậy cấu hình mặc định chỉ chạy 1.000 user.
os.environ['OPENBLAS_NUM_THREADS'] = '1'
print('NumPy ALS đã sẵn sàng — không cần implicit.')

In [ ]:
# Đọc CSV theo chunks: không nạp 5,6 triệu dòng vào RAM cùng lúc.
CORE = ['user_id', 'track_id', 'playcount']

def choose_users(train_path, n_users, seed):
    users = set()
    for chunk in pd.read_csv(train_path, usecols=['user_id'], dtype={'user_id': str}, chunksize=100_000):
        users.update(chunk['user_id'].dropna().astype(str).str.strip())
    users = np.array(sorted(users))
    rng = np.random.default_rng(seed)
    return set(rng.choice(users, min(n_users, len(users)), replace=False))

def load_subset(path, users):
    header = pd.read_csv(path, nrows=0).columns.tolist()
    missing = set(CORE) - set(header)
    assert not missing, f'{path} thiếu cột: {missing}'
    keep = CORE + [c for c in ['engagement_score', 'engagement_level'] if c in header]
    pieces = []
    for chunk in pd.read_csv(path, usecols=keep, dtype={'user_id': str, 'track_id': str}, chunksize=100_000):
        chunk['user_id'] = chunk['user_id'].str.strip()
        selected = chunk[chunk['user_id'].isin(users)]
        if not selected.empty:
            pieces.append(selected)
    x = pd.concat(pieces, ignore_index=True)
    x['track_id'] = x['track_id'].astype(str).str.strip()
    x['playcount'] = pd.to_numeric(x['playcount'], errors='raise')
    assert (x['playcount'] > 0).all() and not x[CORE].isna().any().any()
    assert not x.duplicated(['user_id', 'track_id']).any(), 'Mỗi user-track phải chỉ có một dòng'
    if 'engagement_score' in x:
        x['engagement_score'] = pd.to_numeric(x['engagement_score'], errors='raise')
        assert x['engagement_score'].between(0, 1).all()
    return x.sort_values(['user_id', 'track_id']).reset_index(drop=True)

chosen_users = choose_users(TRAIN_PATH, MAX_TRAIN_USERS, SEED)
train = load_subset(TRAIN_PATH, chosen_users)
test = load_subset(TEST_PATH, chosen_users)
print(f'Train sample: {len(train):,} interactions | {train.user_id.nunique():,} users | {train.track_id.nunique():,} tracks')
print(f'Test sample : {len(test):,} interactions | {test.user_id.nunique():,} users | {test.track_id.nunique():,} tracks')
print('Cột train:', train.columns.tolist())

In [ ]:
# Hàm tách validation, huấn luyện và đánh giá.
def inner_split(frame, seed, ratio=0.15):
    rng, base, valid = np.random.default_rng(seed), [], []
    for _, group in frame.groupby('user_id', sort=True):
        n_valid = max(1, int(len(group) * ratio))
        order = rng.permutation(len(group))
        valid.append(group.iloc[order[:n_valid]])
        base.append(group.iloc[order[n_valid:]])
    return pd.concat(base, ignore_index=True), pd.concat(valid, ignore_index=True)

def confidence(frame, weighting):
    counts = frame.playcount.to_numpy(dtype=float)
    if weighting == 'binary':
        weights = np.ones_like(counts)
    elif weighting == 'log':
        weights = np.log1p(counts)
    elif weighting == 'engagement':
        assert 'engagement_score' in frame, 'train.csv thiếu engagement_score'
        weights = np.log1p(counts) * (0.5 + frame.engagement_score.to_numpy(dtype=float))
    else:
        raise ValueError(weighting)
    return 1.0 + ALS_ALPHA * weights

def fit_als(frame, weighting):
    # NumPy reference implementation of weighted implicit ALS (Hu, Koren & Volinsky).
    users, items = sorted(frame.user_id.unique()), sorted(frame.track_id.unique())
    user_index, item_index = {u:i for i, u in enumerate(users)}, {t:i for i, t in enumerate(items)}
    ui, ti, conf = frame.user_id.map(user_index).to_numpy(), frame.track_id.map(item_index).to_numpy(), confidence(frame, weighting)
    rng = np.random.default_rng(SEED)
    user_factors = rng.normal(0, 0.05, (len(users), FACTORS))
    item_factors = rng.normal(0, 0.05, (len(items), FACTORS))
    user_rows, item_rows = [[] for _ in users], [[] for _ in items]
    for user, item, value in zip(ui, ti, conf):
        user_rows[user].append((item, value))
        item_rows[item].append((user, value))

    def update(target, other, groups):
        gram = other.T @ other
        eye = REGULARIZATION * np.eye(FACTORS)
        for index, group in enumerate(groups):
            indices, values = zip(*group)
            selected = other[list(indices)]
            values = np.asarray(values)
            a = gram + (selected.T * (values - 1)) @ selected + eye
            b = selected.T @ values
            target[index] = np.linalg.solve(a, b)

    for _ in range(ITERATIONS):
        update(user_factors, item_factors, user_rows)
        update(item_factors, user_factors, item_rows)
    return {'name': 'als_' + weighting, 'users': users, 'items': items, 'uix': user_index, 'iix': item_index,
            'seen': frame.groupby('user_id').track_id.agg(set).to_dict(),
            'user_factors': user_factors, 'item_factors': item_factors}

def fit_popularity(frame):
    return {'name': 'popularity', 'items': sorted(frame.track_id.unique()),
            'seen': frame.groupby('user_id').track_id.agg(set).to_dict(),
            'scores': frame.groupby('track_id').user_id.nunique().to_dict()}

def recommend(model, user, k=50):
    seen = model['seen'].get(user, set())
    if model['name'] == 'popularity':
        ranked = sorted((t for t in model['items'] if t not in seen), key=lambda t: (-model['scores'][t], t))[:k]
        return ranked, [float(model['scores'][t]) for t in ranked]
    vector = model['user_factors'][model['uix'][user]]
    scores = model['item_factors'] @ vector
    for track in seen:
        scores[model['iix'][track]] = -np.inf
    take = min(k, len(model['items']) - len(seen))
    chosen = np.argpartition(-scores, take - 1)[:take]
    chosen = sorted(chosen, key=lambda i: (-scores[i], model['items'][i]))
    return [model['items'][i] for i in chosen], [float(scores[i]) for i in chosen]

def metric_row(ranking, truth, k):
    hits = [int(track in truth) for track in ranking[:k]]
    recall = sum(hits) / len(truth)
    dcg = sum(hit / math.log2(rank + 2) for rank, hit in enumerate(hits))
    ideal = sum(1 / math.log2(rank + 2) for rank in range(min(k, len(truth))))
    found = ap = 0.0
    for rank, hit in enumerate(hits, 1):
        found += hit
        ap += hit * found / rank
    return recall, dcg / ideal, ap / min(k, len(truth))

def evaluate_model(model, heldout, max_users, seed, save_recommendations=False):
    catalogue = set(model['items'])
    eligible = heldout[heldout.track_id.isin(catalogue)].copy()
    truth = eligible.groupby('user_id').track_id.agg(set).to_dict()
    users = np.array(sorted(truth))
    if max_users and len(users) > max_users:
        users = np.random.default_rng(seed).choice(users, max_users, replace=False)
    rows, recs = [], []
    for user in sorted(users):
        ranking, scores = recommend(model, user, 50)
        r10, n10, m10 = metric_row(ranking, truth[user], 10)
        r50, n50, m50 = metric_row(ranking, truth[user], 50)
        rows.append({'user_id': user, 'Recall@10': r10, 'NDCG@10': n10, 'MAP@10': m10, 'Recall@50': r50, 'NDCG@50': n50, 'MAP@50': m50})
        if save_recommendations:
            recs.extend({'user_id': user, 'track_id': t, 'rank': rank, 'score': score} for rank, (t, score) in enumerate(zip(ranking, scores), 1))
    detail = pd.DataFrame(rows)
    summary = detail.drop(columns='user_id').mean().to_dict()
    summary.update(n_users_evaluated=len(detail), excluded_unseen_item_positives=len(heldout) - len(eligible))
    return summary, detail, pd.DataFrame(recs), truth, list(users)

inner_train, validation = inner_split(train, SEED)
print(f'Inner train: {len(inner_train):,}; validation: {len(validation):,}')

In [ ]:
# Validation chọn mô hình; sau đó huấn luyện lại trên toàn bộ train và test đúng một lần.
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True)

all_rows, best_name, best_ndcg = [], None, -np.inf
validation_models = [fit_popularity(inner_train)] + [fit_als(inner_train, w) for w in WEIGHTINGS]
for model in validation_models:
    summary, detail, _, _, _ = evaluate_model(model, validation, MAX_EVAL_USERS, SEED)
    summary.update(stage='validation', model=model['name'])
    all_rows.append(summary)
    detail.to_csv(OUT_DIR / f"validation_{model['name']}_per_user.csv", index=False)
    if model['name'].startswith('als_') and summary['NDCG@10'] > best_ndcg:
        best_name, best_ndcg = model['name'], summary['NDCG@10']

selected_weighting = best_name.removeprefix('als_')
selected_config = {'weighting': selected_weighting, 'factors': FACTORS, 'regularization': REGULARIZATION,
                   'als_alpha': ALS_ALPHA, 'iterations': ITERATIONS, 'seed': SEED}
(OUT_DIR / 'selected_config.json').write_text(json.dumps(selected_config, indent=2), encoding='utf-8')
print('Selected on validation:', selected_config)

final_models = [fit_popularity(train), fit_als(train, selected_weighting)]
for model in final_models:
    summary, detail, recommendations, truth, users = evaluate_model(model, test, MAX_EVAL_USERS, SEED, save_recommendations=True)
    output_name = 'als_selected' if model['name'].startswith('als_') else 'popularity'
    summary.update(stage='test', model=output_name)
    all_rows.append(summary)
    detail.to_csv(OUT_DIR / f'test_{output_name}_per_user.csv', index=False)
    recommendations.to_csv(OUT_DIR / f'test_{output_name}_recommendations.csv', index=False)

metrics = pd.DataFrame(all_rows)
metrics.to_csv(OUT_DIR / 'metrics.csv', index=False)
manifest = {'data_dir': str(DATA_DIR), 'seed': SEED, 'max_train_users': MAX_TRAIN_USERS, 'max_eval_users': MAX_EVAL_USERS,
            'train_rows': len(train), 'test_rows': len(test), 'train_users': int(train.user_id.nunique()),
            'selected_by': 'Validation NDCG@10 from train.csv only', 'official_test_used_for_selection': False}
(OUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

display(metrics[['stage', 'model', 'Recall@10', 'NDCG@10', 'MAP@10', 'Recall@50', 'NDCG@50', 'MAP@50', 'n_users_evaluated']].round(4))
print('Output folder:', OUT_DIR)

In [ ]:
# Biểu đồ và danh sách file để tải từ tab Output.
import matplotlib.pyplot as plt
test_metrics = metrics[(metrics.stage == 'test') & (metrics.model.isin(['popularity', 'als_selected']))].set_index('model')
score_cols = ['Recall@10', 'NDCG@10', 'MAP@10']
test_scores = test_metrics[score_cols].apply(pd.to_numeric, errors='coerce')
assert not test_scores.isna().any().any(), 'Thiếu metric test; hãy chạy lại cell thí nghiệm.'
test_scores.plot.bar(figsize=(9, 4), rot=0, ylabel='Score', title='Test: Popularity vs ALS được chọn')
plt.ylim(bottom=0); plt.tight_layout(); plt.show()

ratio = (test_scores.loc['als_selected'] / test_scores.loc['popularity'])
display(ratio.rename('ALS / Popularity').round(2).to_frame())

print('Các file để tải:')
for file in sorted(OUT_DIR.iterdir()):
    print('-', file.name)